# Comprender los datos

## 1. Contexto y estructura

1. ¿Qué fenomeno registra la base?\
La base registra los delitos denunciados o conocidos por el departamento de policía de Los Angeles (LAPD), California durante el periodo 2020 a 2024.


2. ¿Quién la produjo y para qué?\
Si bien el registro pertenece al departamento de la policía de Los Angeles, el archivo se encuentra publicado por el gobierno de la ciudad mediante su portal de datos abiertos. Dentro de la página no hay una justificación de por qué se encuentra pública; sin embargo, dentro del ecosistema de datos abiertos, conocer la ocurrencia y ubicación de los delitos constituye un ejercicio de transparencia valioso para la ciudadanía, la academia y el sector empresarial.


3. ¿Cuántas filas y columnas contiene?\
El archivo contiene: 
- **filas:** 1,004,895
- **columnas:** 28


4. ¿Cuál es su cobertura temoral/geográfica?\
El archivo contiene delitos reportados entre 2020 y 2024. 
Su cobertura geográfica abarca 21 áreas geográficas de la ciudad de Los Ángeles, California.


5. ¿Qué representa exactamente una fila (unidad de observación)?\
Cada fila representa un incidente delictivo (crime incident).


6. ¿Qué entidades, eventos e identificadores existen?\
La entidad corresponde al incidente delictivo reportado o conocido por LAPD. Cada incidente puede contener 1 o más delitos, registrados según su gravedad. El archivo cuenta con la columna DR_NO la cual representa el número de registro de la división. Este número contiene los dos dígitos del año, el identificador del área donde ocurrió el delito, y otros cinco dígitos. Asimismo, el archivo contiene un identificador para el área donde ocurrió el delito, representado por la columna 'AREA'. Por último, el archivo cuenta con la variable 'Crm Cd', la cual indica de manera numérica el tipo de delito reportado.


7. ¿Un identificador repetido representa necesariamente un error?\
Al explorar la base, no se encontró que para la variable 'DR_NO' existan registros repetidos. De haberlo, sí sería un error porque, conforme a la documentación de la base, cada fila representa a un único incidente. Para las variables 'AREA' y 'Crm Cd' sí puede repetirse, ya que cada delito pudo haberse cometido en una misma área de la ciudad y también un mismo delito. 


## 2. Calidad y decisiones de limpieza

8. Tipos de datos, faltantes y posibles valores inválidos.\
Para el esquema clean, decidimos trabajar con las siguientes variables:
- **fecha_occ:** TIMESTAMP
- **hora_occ:** INTEGER
- **diasem:** INTEGER
- **anio:** INTEGER
- **num_area:** INTEGER
- **nom_area:** VARCHAR
- **delito:** VARCHAR
- **sexo_vic:** VARCHAR
- **edad_vic:** INTEGER

Al explorar la base, encontramos algunos datos faltantes y valores inválidos, específicamente para las variables sexo de la víctima (sexo_vic) y edad de la víctima (edad_vic). 
Para sexo_vic, la documentación especifica que los valores de este campo son: F-female, M-male, y X-unknown. Sin embargo, aparecen otros valores como: '-' (1), 'H' (144) y 'NULL' (144,631).

Para la variable edad_vic, hay valores negativos que van del -4 al -1 (137), 120 (1). Igualmente, llama la atención la alta frecuencia del valor cero (269,178). La documentación no refiere sobre este campo en particular, por lo que dichos valores se tomarán como inválidos, ya que no corresponden con el tipo de dato o bien su frecuencia es atípica con respecto al resto de las edades registradas en el archivo. 

Finalmente, al revisar el número de registros por año, se evidencia una caída significativa en el número de registros para el año 2024, especialmente a partir de marzo. Al revisar la documentación del archivo, detectamos que el LAPD transitó a un diferente esquema de registro de incidentes delictivos (UCR a NIBRS). No obstante, no hay aclaraciones sobre por qué unos incidentes sí fueron registrados y otros no. Por esta razón, decidimos acotar el análisis para los registros que tuvieran como año de ocurrencia entre 2020 y 2023. 


9. Conversión de fechas y números; códigos y categorías inconsistentes.\ 
En el esquema clean construimos un solo campo de fecha de ocurrencia ('fecha_occ') que junta la fecha como la hora de ocurrencia del incidente delictivo. Esto se hizo mediante la siguiente sintaxis: 

```sql
(strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p')::DATE
    + strptime(lpad("TIME OCC", 4, '0'), '%H%M')::TIME) AS fecha_occ,
```

Asimismo, construimos dos campos derivados a partir de fecha_occ: día de la semana (diasem) y año (anio). 
La sintaxis para construir dichos campos fue la siguiente:

```sql
    EXTRACT(ISODOW FROM strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p'))::INTEGER AS diasem,
    EXTRACT(YEAR FROM strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p'))::INTEGER AS anio,
```

Para la variable de sexo_vict, se empleó la siguiente sintaxis a fin de visibilizar aquellos registros con los valores documentados por la base (F,M,X):

```sql
CASE WHEN "Vict Sex" IN ('F', 'M', 'X') THEN "Vict Sex" END AS sexo_vic,
```

Para la variable de edad_vic, se empleó la siguiente sintaxis:
```sql
CASE WHEN "Vict Age"::INTEGER BETWEEN 1 AND 99 THEN "Vict Age"::INTEGER END AS edad_vic,
```

En los registros que tengan valores diferentes a los especificados en la documentación para las variables sex_vic o edad_vic, aparecerá el valor 'NULL'. 
Así se mantienen todos los registros, pero a la vez se depura de mejor manera cuáles sí deben aparecer en las visualizaciones y cuáles no.


10. Posibles duplicados y reglas para tratarlos, cuando corresponda. 
En este caso no encontramos registros duplicados, por lo que no aplica un tratamiento distinto. 


11. Evidencia de las decisiones aplicadas en sql/02_clean.sql


**sexo_vic**\
delitos_LA D SELECT "Vict Sex", COUNT(*) AS n FROM raw.delitos_LA GROUP BY "Vict Sex" ORDER BY "Vict Sex";

```text
┌──────────┬────────┐
│ Vict Sex │   n    │
│ varchar  │ int64  │
├──────────┼────────┤
│ -        │      1 │
│ F        │ 358553 │
│ H        │    114 │
│ M        │ 403842 │
│ X        │  97753 │
│ NULL     │ 144631 │
└──────────┴────────┘
```

**edad_vic**\
delitos_LA D SELECT "Vict Age", COUNT(*) AS n FROM raw.delitos_LA GROUP BY "Vict Age" ORDER BY "Vict Age";

```text
┌──────────┬────────┐
│ Vict Age │   n    │
│ varchar  │ int64  │
├──────────┼────────┤
│ -1       │    100 │
│ -2       │     28 │
│ -3       │      6 │
│ -4       │      3 │
│ 0        │ 269178 │
│ 10       │    795 │
│ 11       │   1232 │
│ 12       │   1784 │
│ 120      │      1 │
│ 13       │   2498 │
│ 14       │   2972 │
│ 15       │   3562 │
│ 16       │   3956 │
│ 17       │   4279 │
│ 18       │   6073 │
│ 19       │   9918 │
│ 2        │    429 │
│ 20       │  12004 │
│ 21       │  14447 │
│ 22       │  16671 │
│ ·        │     ·  │
│ ·        │     ·  │
│ ·        │     ·  │
│ 81       │    969 │
│ 82       │    897 │
│ 83       │    825 │
│ 84       │    620 │
│ 85       │    607 │
│ 86       │    501 │
│ 87       │    381 │
│ 88       │    352 │
│ 89       │    308 │
│ 9        │    697 │
│ 90       │    280 │
│ 91       │    231 │
│ 92       │    175 │
│ 93       │    124 │
│ 94       │    105 │
│ 95       │    100 │
│ 96       │     94 │
│ 97       │     72 │
│ 98       │     71 │
│ 99       │    354 │
└──────────┴────────┘
```

**anio**\
SELECT EXTRACT(YEAR FROM strptime("DATE OCC", '%m/%d/%Y %I:%M:%S %p')) AS anio,\
                    COUNT(*) AS n,\
                    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM raw.delitos_LA), 2) AS pct\
             FROM raw.delitos_LA\
             GROUP BY anio\
             ORDER BY anio;\

```text
┌───────┬────────┬────────┐
│ anio  │   n    │  pct   │
│ int64 │ int64  │ double │
├───────┼────────┼────────┤
│  2020 │ 199847 │  19.89 │
│  2021 │ 209876 │  20.89 │
│  2022 │ 235259 │  23.41 │
│  2023 │ 232345 │  23.12 │
│  2024 │ 127567 │  12.69 │
└───────┴────────┴────────┘
```

Para el tema del registro por años, la documentación del archivo contiene lo siguiente:
"On March 7th, 2024, the Los Angeles Police Department (LAPD) adopted a new Records Management System for reporting crimes and arrests. This new system is implemented to comply with the FBI's mandate to collect NIBRS-only data (NIBRS — FBI - https://www.fbi.gov/how-we-can-help-you/more-fbi-services-and-information/ucr/nibrs)."


## 3. Perfilado según el tipo de variable

### 3.1 Categóricas

**sex_vic**

```text
┌───────────┬────────┬────────┐
│ categoria │   n    │  pct   │
│  varchar  │ int64  │ double │
├───────────┼────────┼────────┤
│ M         │ 403842 │  40.19 │
│ F         │ 358553 │  35.68 │
│ X         │  97753 │   9.73 │
│ Faltante  │ 144746 │   14.4 │
└───────────┴────────┴────────┘
```

**nom_area**

```text
┌─────────────┬───────┬────────┐
│  categoria  │   n   │  pct   │
│   varchar   │ int64 │ double │
├─────────────┼───────┼────────┤
│ Central     │ 69668 │   6.93 │
│ 77th Street │ 61756 │   6.15 │
│ Pacific     │ 59513 │   5.92 │
│ Southwest   │ 57434 │   5.72 │
│ Hollywood   │ 52429 │   5.22 │
│ N Hollywood │ 51106 │   5.09 │
│ Olympic     │ 50070 │   4.98 │
│ Southeast   │ 49929 │   4.97 │
│ Newton      │ 49173 │   4.89 │
│ Wilshire    │ 48237 │    4.8 │
│ Rampart     │ 46824 │   4.66 │
│ West LA     │ 45725 │   4.55 │
│ Northeast   │ 42951 │   4.27 │
│ Van Nuys    │ 42881 │   4.27 │
│ West Valley │ 42146 │   4.19 │
│ Devonshire  │ 41743 │   4.15 │
│ Harbor      │ 41392 │   4.12 │
│ Topanga     │ 41365 │   4.12 │
│ Mission     │ 40342 │   4.01 │
│ Hollenbeck  │ 37081 │   3.69 │
│ Foothill    │ 33129 │    3.3 │
└─────────────┴───────┴────────┘
```

**delito**

```text
┌──────────────────────────────────────────────────────────┬────────┬────────┐
│                        categoria                         │   n    │  pct   │
│                         varchar                          │ int64  │ double │
├──────────────────────────────────────────────────────────┼────────┼────────┤
│ VEHICLE - STOLEN                                         │ 115184 │  11.46 │
│ BATTERY - SIMPLE ASSAULT                                 │  74821 │   7.45 │
│ BURGLARY FROM VEHICLE                                    │  63515 │   6.32 │
│ THEFT OF IDENTITY                                        │  62536 │   6.22 │
│ VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)  │  61086 │   6.08 │
│ BURGLARY                                                 │  57871 │   5.76 │
│ THEFT PLAIN - PETTY ($950 & UNDER)                       │  53716 │   5.35 │
│ ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT           │  53523 │   5.33 │
│ INTIMATE PARTNER - SIMPLE ASSAULT                        │  46712 │   4.65 │
│ THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER)          │  41311 │   4.11 │
│ THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER)      │  36940 │   3.68 │
│ THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD │  35149 │    3.5 │
│ ROBBERY                                                  │  32315 │   3.22 │
│ SHOPLIFTING - PETTY THEFT ($950 & UNDER)                 │  30908 │   3.08 │
│ VANDALISM - MISDEAMEANOR ($399 OR UNDER)                 │  25374 │   2.53 │
│ CRIMINAL THREATS - NO WEAPON DISPLAYED                   │  19278 │   1.92 │
│ TRESPASSING                                              │  18424 │   1.83 │
│ BRANDISH WEAPON                                          │  14532 │   1.45 │
│ INTIMATE PARTNER - AGGRAVATED ASSAULT                    │  12656 │   1.26 │
│ VIOLATION OF RESTRAINING ORDER                           │  11748 │   1.17 │
│ LETTERS, LEWD  -  TELEPHONE CALLS, LEWD                  │   8712 │   0.87 │
└──────────────────────────────────────────────────────────┴────────┴────────┘
```

**num_area**

```text
┌───────────┬───────┬────────┐
│ categoria │   n   │  pct   │
│  varchar  │ int64 │ double │
├───────────┼───────┼────────┤
│ 1         │ 69668 │   6.93 │
│ 2         │ 46824 │   4.66 │
│ 3         │ 57434 │   5.72 │
│ 4         │ 37081 │   3.69 │
│ 5         │ 41392 │   4.12 │
│ 6         │ 52429 │   5.22 │
│ 7         │ 48237 │    4.8 │
│ 8         │ 45725 │   4.55 │
│ 9         │ 42881 │   4.27 │
│ 10        │ 42146 │   4.19 │
│ 11        │ 42951 │   4.27 │
│ 12        │ 61756 │   6.15 │
│ 13        │ 49173 │   4.89 │
│ 14        │ 59513 │   5.92 │
│ 15        │ 51106 │   5.09 │
│ 16        │ 33129 │    3.3 │
│ 17        │ 41743 │   4.15 │
│ 18        │ 49929 │   4.97 │
│ 19        │ 40342 │   4.01 │
│ 20        │ 50070 │   4.98 │
│ 21        │ 41365 │   4.12 │
└───────────┴───────┴────────┘
```

**diasem**

```text
┌───────────┬────────┬────────┐
│ categoria │   n    │  pct   │
│  varchar  │ int64  │ double │
├───────────┼────────┼────────┤
│ 1         │ 141532 │  14.08 │
│ 2         │ 138125 │  13.75 │
│ 3         │ 142688 │   14.2 │
│ 4         │ 141792 │  14.11 │
│ 5         │ 153663 │  15.29 │
│ 6         │ 147457 │  14.67 │
│ 7         │ 139637 │   13.9 │
└───────────┴────────┴────────┘
```

**hora_occ**

```text
┌───────────┬───────┬────────┐
│ categoria │   n   │  pct   │
│  varchar  │ int64 │ double │
├───────────┼───────┼────────┤
│ 0         │ 40467 │   4.03 │
│ 1         │ 29760 │   2.96 │
│ 2         │ 25214 │   2.51 │
│ 3         │ 22190 │   2.21 │
│ 4         │ 18756 │   1.87 │
│ 5         │ 17288 │   1.72 │
│ 6         │ 23184 │   2.31 │
│ 7         │ 26264 │   2.61 │
│ 8         │ 37243 │   3.71 │
│ 9         │ 36519 │   3.63 │
│ 10        │ 43020 │   4.28 │
│ 11        │ 43663 │   4.35 │
│ 12        │ 67806 │   6.75 │
│ 13        │ 45572 │   4.54 │
│ 14        │ 49295 │   4.91 │
│ 15        │ 52811 │   5.26 │
│ 16        │ 52968 │   5.27 │
│ 17        │ 58809 │   5.85 │
│ 18        │ 59955 │   5.97 │
│ 19        │ 55596 │   5.53 │
│ 20        │ 56349 │   5.61 │
│ 21        │ 50788 │   5.05 │
│ 22        │ 49100 │   4.89 │
│ 23        │ 42277 │   4.21 │
└───────────┴───────┴────────┘
```

**anio**

```text
┌───────────┬────────┬────────┐
│ categoria │   n    │  pct   │
│  varchar  │ int64  │ double │
├───────────┼────────┼────────┤
│ 2020      │ 199847 │  19.89 │
│ 2021      │ 209876 │  20.89 │
│ 2022      │ 235259 │  23.41 │
│ 2023      │ 232345 │  23.12 │
│ 2024      │ 127567 │  12.69 │
└───────────┴────────┴────────┘
```

### 3.2 Numéricas

**edad_vic**

```text
┌─────────┬─────────┬───────────┬────────┬─────────┬───────┬────────┬────────┬───────┬──────────┬──────────┬────────┬────────┐
│  total  │ validos │ faltantes │ media  │ mediana │ moda  │ minimo │ maximo │ rango │ desv_est │ varianza │   q1   │   q3   │
│  int64  │  int64  │   int64   │ double │ double  │ int32 │ int32  │ int32  │ int32 │  double  │  double  │ double │ double │
├─────────┼─────────┼───────────┼────────┼─────────┼───────┼────────┼────────┼───────┼──────────┼──────────┼────────┼────────┤
│ 1004894 │  735578 │    269316 │  39.51 │    37.0 │    30 │      2 │     99 │    97 │    15.57 │   242.45 │   28.0 │   50.0 │
└─────────┴─────────┴───────────┴────────┴─────────┴───────┴────────┴────────┴───────┴──────────┴──────────┴────────┴────────┘
```

### 3.3 Temporales

**fecha_occ**

```text
┌─────────┬─────────┬───────────┬─────────────────────┬─────────────────────┐
│  total  │ validos │ faltantes │      fecha_min      │      fecha_max      │
│  int64  │  int64  │   int64   │      timestamp      │      timestamp      │
├─────────┼─────────┼───────────┼─────────────────────┼─────────────────────┤
│ 1004894 │ 1004894 │         0 │ 2020-01-01 00:01:00 │ 2024-12-30 23:00:00 │
└─────────┴─────────┴───────────┴─────────────────────┴─────────────────────┘
```

- La granularidad de este campo permite segmentar el análisis por día, mes, año, hora y minuto de ocurrencia. 
- Su cobertura va del 1 de enero de 2020 hasta el 30 de diciembre de 2024. 

### 3.4 Geográficas

La única variable geográfica para este análisis será nom_area que representa el nombre del área donde ocurrió el incidente delictivo. 
Si bien el archivo contiene coordenadas (lat,long), éstas no serán utilizadas porque no se hará un análisis geoespacial de los delitos. 
Así, dicha variable cubre las 21 zonas geográficas que LAPD divide a la ciudad de Los Angeles. 

## 4. Implicaciones para el análisis

##### Preguntas viables
Conforme al esquema que definimos para este ejercicio, las preguntas son de tipo descriptivas. Nos interesa particularmente conocer qué delitos afectan más a mujeres y a hombres. Por la cobertura temporal del archivo, será interesante tomar en cuenta si hubo diferencias entre años, ya que el periodo analizado coincide con la duración de la pandemia por COVID-19. Asimismo, nos interesa identificar si todas las áreas de la ciudad presentan una distribución similar de delitos o si bien hay áreas que concentran más delitos que otras. Lo anterior puede servir como dato orientador para priorizar los recursos de LAPD. 
Otras preguntas que son viables para el estudio son: ¿los delitos más frecuentes para mujeres y hombres se mantuvieron en el mismo lugar entre años? ¿qué días de la semana y horas son los de mayor riesgo para los delitos de mayor frecuencia? 


##### Sesgos de cobertura

Considerando que el análisis se centrará únicamente en aquellos incidentes que tuvieron un correcto registro, es posible que se estén omitiendo para la variable sexo_vic a ciertas víctimas para las cuales no fue posible identificar su sexo. 
Asimismo, la alta frecuencia del valor cero en la variable edad_vic puede potencialmente sesgar el análisis. No hay certeza que efectivamente se trata de delitos para personas menores de un año o si más bien se trata de registros donde no fue posible identificar la edad de la víctima o bien donde no aplicara. 

##### Precauciones al interpretar los resultados

Es importante considerar que el análisis utilizará incidentes delictivos reportados a LAPD. Esto significa que no se trata de la totalidad de delitos ocurridos en Los Angeles. Si se quisiera entender la magnitud real del problema delictivo, entonces se tendría que realizar un análisis a partir de encuestas de victimización con desagregación local. Este análisis únicamente visualiza los patrones de aquellos eventos que fueron del conocimiento de la autoridad, por lo que se debe tomar esta cautela y los hallazgos contrastarlos con los datos de las encuestas de victimización. 